# Tutorial 5: Time Series Analysis

## Platinum XPT/USD Closing Price Forecasting

The aim of this tutorial is to develop a time-series model for the daily closing price of platinum, XPT/USD.

The model is trained using historical platinum closing-price data. A train/test split is used so that the model can be tested on unseen data. This allows the predicted prices to be compared with the actual prices, which act as the ground truth.

The process followed in this notebook is:

1. Load and inspect the platinum closing-price data.
2. Plot the original time series.
3. Test for stationarity.
4. Split the original data into training and testing data.
5. Decompose only the training data.
6. Model the trend, seasonal and remainder components.
7. Forecast the test set and compare with the ground truth.
8. Refit the model using all available data.
9. Forecast 1 trading day and 5 trading days into the future.
10. Discuss whether it would be prudent to trade in platinum based on the model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

## 1. Loading and Preparing the Data

The data file used in this tutorial is `Platinum.csv`.

The file contains the following columns:

- `timestamp`
- `open`
- `high`
- `low`
- `close`
- `volume`

The `close` column is used because the question asks for a model of the daily closing prices of platinum.

The data is sorted from oldest to newest because time-series models depend on the correct chronological order.

In [ ]:
filename = "Platinum.csv"

# Load CSV file
df_raw = pd.read_csv(filename)

print("First five rows of raw data:")
display(df_raw.head())

print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nRaw data information:")
print(df_raw.info())

# Keep only the needed columns
df = df_raw[["timestamp", "close"]].copy()

# Convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Convert close price to numeric
df["close"] = pd.to_numeric(df["close"], errors="coerce")

# Sort from oldest to newest
df = df.sort_values("timestamp").reset_index(drop=True)

# Drop any missing values
df = df.dropna().reset_index(drop=True)

# Rename timestamp to date for easier reading
df = df.rename(columns={"timestamp": "date"})

print("\nPrepared data:")
display(df.head())

print("\nPrepared data summary:")
print("Number of samples:", len(df))
print("First date:", df["date"].iloc[0])
print("Last date:", df["date"].iloc[-1])
print("Minimum close price:", df["close"].min())
print("Maximum close price:", df["close"].max())

# Create arrays for modelling
dates = df["date"]
close = df["close"].to_numpy(dtype=float)

In [ ]:
# Plot the original closing-price series
plt.figure()
plt.plot(dates, close, label="Actual platinum closing price")
plt.xlabel("Date")
plt.ylabel("Closing price [USD]")
plt.title("Platinum XPT/USD Daily Closing Price")
plt.legend()
plt.show()

### Discussion of the Original Closing-Price Signal

The platinum closing-price signal shows a clear change in price level over time. At the start of the dataset, the price is close to 900 USD. The price then increases strongly during 2025 and reaches a peak above 2700 USD near the beginning of 2026.

After the peak, the signal becomes highly volatile and decreases toward approximately 2000 USD near the end of the dataset.

This behaviour shows that the mean of the signal is not constant over time. Therefore, the original closing-price series is likely non-stationary. A stationarity test is required before choosing the final time-series model.

## 3. Stationarity Testing

Before fitting a time-series model, the stationarity of the platinum closing-price signal must be tested.

A stationary signal has statistical properties that remain approximately constant over time. This means that the mean, variance and autocorrelation structure do not change significantly over time.

The Augmented Dickey-Fuller test is used to test for stationarity.

The hypotheses are:

$$
H_0: \text{The time series is non-stationary}
$$

$$
H_1: \text{The time series is stationary}
$$

The decision rule is:

$$
p < 0.05 \Rightarrow \text{Reject } H_0
$$

$$
p \geq 0.05 \Rightarrow \text{Fail to reject } H_0
$$

If the p-value is less than 0.05, the signal is treated as stationary. If the p-value is greater than or equal to 0.05, the signal is treated as non-stationary.

In [ ]:
# ADF test function
def adf_test(series, name):
 """
 Performs the Augmented Dickey-Fuller test and prints the result.
 """
 result = adfuller(series)
 
 print(f"ADF test for: {name}")
 print(f"ADF statistic: {result[0]:.4f}")
 print(f"p-value: {result[1]:.4f}")
 
 print("Critical values:")
 for key, value in result[4].items():
 print(f" {key}: {value:.4f}")
 
 if result[1] < 0.05:
 print("Conclusion: Reject H0. The series is stationary.")
 else:
 print("Conclusion: Fail to reject H0. The series is non-stationary.")
 
 print("-" * 60)


adf_test(close, "Original platinum closing price")


### First Difference of the Closing Price

The original closing-price signal is expected to be non-stationary because the price level changes significantly over time.

To reduce this non-stationary behaviour, the first difference is calculated:

$$
\Delta x_t = x_t - x_{t-1}
$$

where:

$$
x_t = \text{closing price at time } t
$$

$$
x_{t-1} = \text{closing price at the previous time step}
$$

$$
\Delta x_t = \text{daily change in closing price}
$$

The first-difference signal represents the daily price change instead of the absolute price level. This often removes the slow trend and makes the signal more suitable for ARMA-type modelling.

In [ ]:
# First difference and ADF test
diff_close = np.diff(close)
diff_dates = dates.iloc[1:].reset_index(drop=True)

plt.figure()
plt.plot(diff_dates, diff_close, label="First difference")
plt.xlabel("Date")
plt.ylabel("Daily price change [USD]")
plt.title("First Difference of Platinum Closing Price")
plt.legend()
plt.show()

adf_test(diff_close, "First difference of platinum closing price")

### Stationarity Test Discussion

The ADF test on the original platinum closing-price signal gave a p-value of 0.7426.

Since:

$$
0.7426 > 0.05
$$

the null hypothesis is not rejected. This means that the original closing-price signal is non-stationary. This result agrees with the original plot, where the price level changes significantly over time.

The first difference of the closing price was then calculated. The ADF test on the first-difference signal gave a p-value of 0.0000.

Since:

$$
0.0000 < 0.05
$$

the null hypothesis is rejected. This means that the first-difference signal is stationary.

The first-difference plot also supports this result because the signal fluctuates around an approximately constant mean. However, the volatility increases near the end of the dataset, especially around the large price movements in early 2026. This is expected for commodity price data.

## 4. Train/Test Split

The original closing-price series is split into a training set and a testing set.

The training set is used to build the model, while the testing set is used as unseen ground truth to evaluate the prediction performance.

This is important because a model that only fits historical data has not necessarily learned to forecast future values. By forecasting the test set and comparing the forecast with the actual test values, the model can be evaluated more fairly.

An 80/20 split is used:

$$
80\% = \text{training data}
$$

$$
20\% = \text{testing data}
$$

In [ ]:
# Train/test split of the original closing-price series

train_ratio = 0.8
split_index = int(len(close) * train_ratio)

train_dates = dates.iloc[:split_index].reset_index(drop=True)
test_dates = dates.iloc[split_index:].reset_index(drop=True)

train_close = close[:split_index]
test_close = close[split_index:]

t_train = np.arange(len(train_close))
t_test = np.arange(len(train_close), len(train_close) + len(test_close))

print("Total samples:", len(close))
print("Training samples:", len(train_close))
print("Testing samples:", len(test_close))
print("Training start date:", train_dates.iloc[0])
print("Training end date:", train_dates.iloc[-1])
print("Testing start date:", test_dates.iloc[0])
print("Testing end date:", test_dates.iloc[-1])

plt.figure()
plt.plot(train_dates, train_close, label="Training data")
plt.plot(test_dates, test_close, label="Testing data")
plt.axvline(train_dates.iloc[-1], linestyle=":", label="Train/Test split")
plt.xlabel("Date")
plt.ylabel("Closing price [USD]")
plt.title("Train/Test Split of Platinum Closing Price")
plt.legend()
plt.show()

### Train/Test Split Discussion

The dataset contains 418 samples. An 80/20 split was used, giving 334 training samples and 84 testing samples.

The training data covers the period from 1 January 2025 to 26 January 2026. The testing data covers the period from 27 January 2026 to 5 May 2026.

This split is suitable because the model is trained on the earlier historical data and then tested on later unseen data. The testing period is also challenging because it includes the volatile period after the large platinum price peak. Therefore, the test set provides a meaningful way to check whether the model can forecast future prices rather than only fit past values.

## 5. Decomposition of the Training Data

Only the training data is decomposed. This is important because the model must not use information from the test set during training.

The closing-price signal is decomposed into three components:

$$
x_t = T_t + S_t + R_t
$$

where:

$$
x_t = \text{original closing price}
$$

$$
T_t = \text{trend component}
$$

$$
S_t = \text{seasonal component}
$$

$$
R_t = \text{remainder component}
$$

The trend component represents the slow movement in the platinum price level. The seasonal component represents repeating short-term behaviour. The remainder component contains the part of the signal that is not explained by the trend or seasonal components.

STL decomposition is used because it separates the time series into trend, seasonal and remainder components in a clear and practical way.

In [ ]:
# STL decomposition on training data only

period = 7

stl = STL(train_close, period=period, robust=True)
stl_result = stl.fit()

train_trend = stl_result.trend
train_seasonal = stl_result.seasonal
train_remainder = stl_result.resid

plt.figure(figsize=(12, 9))

plt.subplot(4, 1, 1)
plt.plot(train_dates, train_close)
plt.title("Training Closing Price")
plt.ylabel("Price [USD]")

plt.subplot(4, 1, 2)
plt.plot(train_dates, train_trend)
plt.title("Training Trend Component")
plt.ylabel("Trend")

plt.subplot(4, 1, 3)
plt.plot(train_dates, train_seasonal)
plt.title("Training Seasonal Component")
plt.ylabel("Seasonal")

plt.subplot(4, 1, 4)
plt.plot(train_dates, train_remainder)
plt.title("Training Remainder Component")
plt.xlabel("Date")
plt.ylabel("Remainder")

plt.tight_layout()
plt.show()

### Decomposition Discussion

The STL decomposition separates the training closing-price signal into a trend component, seasonal component and remainder component.

The trend component captures the main long-term movement of the platinum price. It shows a gradual increase from the start of 2025 and a much stronger rise near the end of the training period.

The seasonal component is relatively small for most of the training data. However, it becomes larger near the end of the training period because the platinum price becomes more volatile during this time.

The remainder component is mostly centred around zero, which means that the trend and seasonal components explain a large part of the signal. However, the remainder contains large spikes near the end of the training data. These spikes represent sudden price movements that are not fully explained by the trend or seasonal components.

This behaviour is expected for commodity price data because platinum prices can change suddenly due to market events, supply and demand changes and investor behaviour.

## 6. Trend Component Model

The trend component is modelled using a polynomial function.

The general polynomial trend model is:

$$
\hat{T}_t = a_0 + a_1t + a_2t^2 + \dots + a_nt^n
$$

where:

$$
\hat{T}_t = \text{estimated trend at time } t
$$

$$
a_0, a_1, a_2, \dots, a_n = \text{polynomial coefficients}
$$

$$
n = \text{polynomial order}
$$

Different polynomial orders are tested. The polynomial order with the lowest RMSE is selected for the trend model.

In [ ]:
# Trend component model

degrees = [1, 2, 3]
trend_models = {}
trend_errors = {}

for deg in degrees:
 coeffs = np.polyfit(t_train, train_trend, deg=deg)
 fitted_trend = np.polyval(coeffs, t_train)
 
 rmse = np.sqrt(mean_squared_error(train_trend, fitted_trend))
 
 trend_models[deg] = {
 "coeffs": coeffs,
 "fitted": fitted_trend
 }
 
 trend_errors[deg] = rmse

trend_error_table = pd.DataFrame({
 "Polynomial order": list(trend_errors.keys()),
 "Trend RMSE": list(trend_errors.values())
})

display(trend_error_table)

best_trend_degree = min(trend_errors, key=trend_errors.get)
best_trend_coeffs = trend_models[best_trend_degree]["coeffs"]
train_trend_model = trend_models[best_trend_degree]["fitted"]

print("Selected polynomial order:", best_trend_degree)
print("Selected polynomial coefficients:")
print(best_trend_coeffs)

plt.figure()
plt.plot(train_dates, train_trend, label="STL trend")
plt.plot(train_dates, train_trend_model, "--", label=f"Polynomial trend model, order {best_trend_degree}")
plt.xlabel("Date")
plt.ylabel("Trend value [USD]")
plt.title("Trend Component Model")
plt.legend()
plt.show()


### Trend Model Discussion

Polynomial trend models of order 1, 2 and 3 were tested. The third-order polynomial gave the lowest trend RMSE of 110.76 USD, so it was selected as the trend model.

The selected polynomial trend model captures the general upward movement in the platinum price. However, it does not perfectly follow the sharp increase near the end of the training period. This shows that the trend model captures the long-term movement, but it cannot fully represent sudden rapid changes in price.

This is acceptable because sudden commodity price movements are difficult to model using a simple deterministic trend model. The remaining short-term behaviour is handled by the seasonal and remainder models.

## 7. Seasonal Component Model

The seasonal component is modelled using an average seasonal cycle.

Since a weekly period was used during STL decomposition, the seasonal period is:

$$
P = 7
$$

Instead of using only the last 7 seasonal values, the average seasonal pattern is calculated over all complete cycles in the training data. This is more stable because the last few values occur during a highly volatile period and may not represent the normal seasonal behaviour.

The seasonal forecast is written as:

$$
\hat{S}_{t+h} = \bar{S}_{j}
$$

where:

$$
j = (t+h) \mod P
$$

and:

$$
\bar{S}_{j} = \text{average seasonal value at position } j \text{ in the cycle}
$$

This gives a more stable seasonal model for forecasting.

In [ ]:
# Improved seasonal component model


# Calculate the average seasonal pattern over the training data
seasonal_pattern = np.zeros(period)

for i in range(period):
 seasonal_pattern[i] = np.mean(train_seasonal[i::period])

# Build the fitted seasonal model over the training set
train_seasonal_model = np.tile(
 seasonal_pattern,
 int(np.ceil(len(train_close) / period))
)[:len(train_close)]

plt.figure()
plt.plot(train_dates, train_seasonal, label="STL seasonal component")
plt.plot(train_dates, train_seasonal_model, "--", label="Average repeated seasonal model")
plt.xlabel("Date")
plt.ylabel("Seasonal value [USD]")
plt.title("Seasonal Component Model")
plt.legend()
plt.show()

print("Average seasonal pattern used for forecasting:")
print(seasonal_pattern)

### Seasonal Model Discussion

The seasonal component was modelled using the average seasonal pattern over all complete 7-day cycles in the training data.

The average seasonal pattern is relatively small compared with the full platinum closing price. This shows that the repeated seasonal behaviour is not the dominant part of the signal. The main movement in the platinum price is still caused by the trend and the remainder components.

Using the average seasonal pattern is more stable than using only the final 7 seasonal values. The final part of the training data is highly volatile, so using only the last 7 values would overestimate the seasonal effect. The average seasonal model gives a more reasonable short-term seasonal estimate.

## 8. Remainder Component Analysis

After modelling the trend and seasonal components, the remainder component is analysed.

The remainder is calculated as:

$$
R_t = x_t - T_t - S_t
$$

where:

$$
R_t = \text{remainder component}
$$

$$
x_t = \text{training closing price}
$$

$$
T_t = \text{trend component}
$$

$$
S_t = \text{seasonal component}
$$

The remainder contains the short-term movement that is not explained by the trend or seasonal components. This component is modelled using an ARMA/ARIMA model.

In [ ]:
# Remainder analysis
plt.figure()
plt.plot(train_dates, train_remainder, label="Training remainder")
plt.xlabel("Date")
plt.ylabel("Remainder [USD]")
plt.title("Training Remainder Component")
plt.legend()
plt.show()

adf_test(train_remainder, "Training remainder component")

## 9. ACF and PACF of the Remainder

The autocorrelation function, ACF, and partial autocorrelation function, PACF, are used to inspect the structure of the remainder component.

The ACF shows how strongly the remainder is correlated with past values of itself.

The PACF shows the direct relationship between the current value and previous lag values after removing the effect of shorter lags.

These plots help motivate the use of an ARMA/ARIMA model for the remainder component.

In [ ]:
# ACF and PACF of training remainder

max_lags = min(40, len(train_remainder)//2 - 1)


plot_acf(train_remainder, lags=max_lags)
plt.title("ACF of Training Remainder")
plt.show()


plot_pacf(train_remainder, lags=max_lags, method="ywm")
plt.title("PACF of Training Remainder")
plt.show()

## 10. ARMA/ARIMA Model for the Remainder

The remainder component is modelled using an ARMA/ARIMA model.

Since the remainder component should already be approximately stationary after removing the trend and seasonal components, the differencing order is chosen as:

$$
d = 0
$$

Therefore:

$$
ARIMA(p,0,q) = ARMA(p,q)
$$

The general ARMA model is:

$$
R_t = c + \phi_1R_{t-1} + \phi_2R_{t-2} + \dots + \phi_pR_{t-p}
+ e_t + \theta_1e_{t-1} + \theta_2e_{t-2} + \dots + \theta_qe_{t-q}
$$

where:

$$
p = \text{autoregressive order}
$$

$$
q = \text{moving-average order}
$$

The best model is selected using the Akaike Information Criterion:

$$
AIC = 2k - 2\ln(L)
$$

where:

$$
k = \text{number of model parameters}
$$

$$
L = \text{maximum likelihood of the model}
$$

The model with the lowest AIC is selected because it gives the best balance between model fit and model complexity.

In [ ]:
# Select ARMA/ARIMA model using AIC

best_aic = np.inf
best_order = None
best_fit = None
results = []

for p in range(0, 5):
 for q in range(0, 5):
 try:
 model = ARIMA(train_remainder, order=(p, 0, q))
 fit = model.fit()
 
 results.append({
 "p": p,
 "d": 0,
 "q": q,
 "AIC": fit.aic,
 "BIC": fit.bic
 })
 
 if fit.aic < best_aic:
 best_aic = fit.aic
 best_order = (p, 0, q)
 best_fit = fit
 
 except:
 continue

results_table = pd.DataFrame(results).sort_values("AIC").reset_index(drop=True)

print("Top models sorted by AIC:")
display(results_table.head(10))

print("Selected ARIMA order:", best_order)
print("Best AIC:", best_aic)
print(best_fit.summary())

### Remainder and ARMA/ARIMA Model Discussion

The remainder component is mostly centred around zero, which means that the trend and seasonal components removed much of the long-term structure from the original closing-price signal.

The ADF test on the training remainder gave a p-value of 0.0000.

Since:

$$
0.0000 < 0.05
$$

the null hypothesis is rejected. Therefore, the training remainder can be treated as stationary.

The ACF plot shows strong correlation at the first few lags, especially lag 1 and lag 2. This indicates that previous remainder values still contain useful information for predicting the current remainder value.

The PACF plot also shows significant values at some early lags. This supports the use of an ARMA/ARIMA model for the remainder component.

Several ARMA models were tested using different values of \(p\) and \(q\). The model with the lowest AIC was selected. The selected model was:

$$
ARIMA(4,0,4)
$$

Since:

$$
d = 0
$$

this is equivalent to an:

$$
ARMA(4,4)
$$

model. This model was selected because it gave the lowest AIC value among the tested candidate models.

## 11. Residual Validation

After fitting the ARMA/ARIMA model, the residuals are analysed.

The residuals are calculated as:

$$
e_t = R_t - \hat{R}_t
$$

where:

$$
e_t = \text{residual}
$$

$$
R_t = \text{actual remainder}
$$

$$
\hat{R}_t = \text{estimated remainder}
$$

A good model should leave residuals that are approximately uncorrelated. This means that most of the predictable structure in the remainder component has been captured.

The Ljung-Box test is used to check for remaining autocorrelation in the residuals.

The hypotheses are:

$$
H_0: \text{The residuals are independently distributed}
$$

$$
H_1: \text{The residuals are not independently distributed}
$$

The decision rule is:

$$
p < 0.05 \Rightarrow \text{Reject } H_0
$$

$$
p \geq 0.05 \Rightarrow \text{Fail to reject } H_0
$$

In [ ]:

# Residual validation
residuals = best_fit.resid

plt.figure()
plt.plot(residuals)
plt.xlabel("Sample")
plt.ylabel("Residual")
plt.title("Residuals of Selected ARMA/ARIMA Remainder Model")
plt.show()


plot_acf(residuals, lags=max_lags)
plt.title("ACF of Remainder Model Residuals")
plt.show()

ljung_box = acorr_ljungbox(residuals, lags=[10], return_df=True)

print("Ljung-Box test:")
display(ljung_box)

if ljung_box["lb_pvalue"].iloc[0] > 0.05:
 print("Conclusion: Fail to reject H0. Residuals are approximately uncorrelated.")
else:
 print("Conclusion: Reject H0. Some autocorrelation remains in the residuals.")

plt.figure()
plt.hist(residuals, bins=25)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Histogram of Remainder Model Residuals")
plt.show()

### Residual Validation Discussion

The residual plot shows that most residuals are close to zero, but there are some large spikes near the end of the training period. These spikes occur during the highly volatile platinum price period and represent sudden price movements that are difficult for a linear ARMA/ARIMA model to capture perfectly.

The ACF plot of the residuals shows that most autocorrelation values lie within the confidence bounds. This suggests that most of the predictable correlation structure has been removed by the model.

The Ljung-Box test gave a p-value of 0.354473.

Since:

$$
0.354473 > 0.05
$$

the null hypothesis is not rejected. Therefore, the residuals can be treated as approximately uncorrelated.

The histogram shows that most residuals are centred near zero, but a few large outliers are present. This is expected for financial commodity data because sudden market changes can cause large price movements.

## 12. Improved Forecasting Model Using ARIMA on the Closing Price

The component-based model showed that polynomial trend extrapolation can become unstable when forecasting into the test period. This happened because the training period ended during a strong upward movement, while the test period moved downward.

To obtain a more reliable forecast, an ARIMA model is fitted directly to the platinum closing price.

The ADF test showed that the original closing-price signal is non-stationary, while the first-difference signal is stationary. Therefore, the ARIMA model uses one order of differencing:

$$
d = 1
$$

The ARIMA model is written as:

$$
ARIMA(p,1,q)
$$

where:

$$
p = \text{autoregressive order}
$$

$$
d = \text{differencing order}
$$

$$
q = \text{moving-average order}
$$

The model is trained only on the training data. It is then used to forecast the test set, which acts as unseen ground truth.

In [ ]:
# Select ARIMA(p,1,q) model on training closing prices


best_aic_price = np.inf
best_order_price = None
best_price_fit = None
price_results = []

for p in range(0, 6):
 for q in range(0, 6):
 try:
 model = ARIMA(train_close, order=(p, 1, q))
 fit = model.fit()
 
 price_results.append({
 "p": p,
 "d": 1,
 "q": q,
 "AIC": fit.aic,
 "BIC": fit.bic
 })
 
 if fit.aic < best_aic_price:
 best_aic_price = fit.aic
 best_order_price = (p, 1, q)
 best_price_fit = fit
 
 except:
 continue

price_results_table = pd.DataFrame(price_results).sort_values("AIC").reset_index(drop=True)

print("Top ARIMA models for closing price sorted by AIC:")
display(price_results_table.head(10))

print("Selected ARIMA order:", best_order_price)
print("Best AIC:", best_aic_price)
print(best_price_fit.summary())

## 13. Walk-Forward Test Forecast

A walk-forward forecast is used to evaluate the model on the test set.

In this method, the model predicts the next value, then the actual value is added to the history before predicting the next step. This is a realistic way to test 1-day-ahead forecasting performance.

The prediction error is:

$$
e_t = x_t - \hat{x}_t
$$

where:

$$
x_t = \text{actual test closing price}
$$

$$
\hat{x}_t = \text{forecasted test closing price}
$$

In [ ]:

# Walk-forward 1-step-ahead forecast on test set


history = list(train_close)
walk_forward_forecast = []

for i in range(len(test_close)):
 model = ARIMA(history, order=best_order_price)
 fit = model.fit()
 
 # Forecast one step ahead
 yhat = fit.forecast(steps=1)[0]
 walk_forward_forecast.append(yhat)
 
 # Add the actual observed test value to history
 history.append(test_close[i])

walk_forward_forecast = np.array(walk_forward_forecast)

walk_forward_comparison = pd.DataFrame({
 "Date": test_dates,
 "Actual close": test_close,
 "Forecast close": walk_forward_forecast,
 "Error": test_close - walk_forward_forecast,
 "Absolute error": np.abs(test_close - walk_forward_forecast),
 "Percentage error": np.abs((test_close - walk_forward_forecast) / test_close) * 100
})

display(walk_forward_comparison.head(10))
display(walk_forward_comparison.tail(10))

In [ ]:
# Plot walk-forward forecast against actual test data


plt.figure()
plt.plot(train_dates, train_close, label="Training data")
plt.plot(test_dates, test_close, label="Actual test data")
plt.plot(test_dates, walk_forward_forecast, "--", label="Walk-forward forecast")
plt.axvline(train_dates.iloc[-1], linestyle=":", label="Train/Test split")
plt.xlabel("Date")
plt.ylabel("Closing price [USD]")
plt.title("Walk-Forward Train/Test Forecast Validation")
plt.legend()
plt.show()

plt.figure()
plt.plot(test_dates, test_close, label="Actual test data")
plt.plot(test_dates, walk_forward_forecast, "--", label="Walk-forward forecast")
plt.xlabel("Date")
plt.ylabel("Closing price [USD]")
plt.title("Zoomed Walk-Forward Test Forecast Validation")
plt.legend()
plt.show()

## 14. Quantitative Test Performance

The walk-forward test forecast is evaluated using MAE, RMSE and MAPE.

The Mean Absolute Error is:

$$
MAE = \frac{1}{N}\sum_{t=1}^{N}|x_t - \hat{x}_t|
$$

The Root Mean Squared Error is:

$$
RMSE = \sqrt{\frac{1}{N}\sum_{t=1}^{N}(x_t - \hat{x}_t)^2}
$$

The Mean Absolute Percentage Error is:

$$
MAPE = \frac{100}{N}\sum_{t=1}^{N}\left|\frac{x_t-\hat{x}_t}{x_t}\right|
$$

In [ ]:
# Walk-forward test performance metrics


wf_mae = mean_absolute_error(test_close, walk_forward_forecast)
wf_rmse = np.sqrt(mean_squared_error(test_close, walk_forward_forecast))
wf_mape = np.mean(np.abs((test_close - walk_forward_forecast) / test_close)) * 100

wf_metrics_table = pd.DataFrame({
 "Metric": ["MAE", "RMSE", "MAPE"],
 "Value": [wf_mae, wf_rmse, wf_mape],
 "Unit": ["USD", "USD", "%"]
})

display(wf_metrics_table)

print(f"Walk-forward MAE: ${wf_mae:.2f}")
print(f"Walk-forward RMSE: ${wf_rmse:.2f}")
print(f"Walk-forward MAPE: {wf_mape:.2f}%")

### Walk-Forward Forecast Discussion

The first component-based test forecast performed poorly because the polynomial trend continued increasing into the test period. The actual test data decreased after the peak, so the extrapolated trend caused the forecast to move too high.

The improved ARIMA model avoids this problem by modelling the closing-price series with one order of differencing. This is justified by the ADF tests, which showed that the original closing-price signal was non-stationary, while the first-difference signal was stationary.

The selected model for the platinum closing-price series was:

$$
ARIMA(5,1,5)
$$

The differencing order is:

$$
d = 1
$$

The walk-forward forecast is a fairer test of practical 1-day-ahead prediction because each forecast is made using only the information available up to that point. The predicted values are then compared with the actual test values.

The quantitative test results were:

$$
MAE = 66.61 \text{ USD}
$$

$$
RMSE = 98.37 \text{ USD}
$$

$$
MAPE = 3.23\%
$$

The MAPE value means that, on average, the model's forecast error was approximately 3.23% of the actual platinum closing price during the test period.

This is a reasonable result for commodity price forecasting, especially because the test period contains volatile price movements after the large peak in early 2026.

## 15. Final Forecast Using All Available Data

After validating the model on the test set, the ARIMA model is refitted using all available platinum closing-price data.

This is done because the final forecast should use as much historical information as possible.

The selected model is:

$$
ARIMA(5,1,5)
$$

The model is used to forecast the platinum closing price 1 trading day and 5 trading days into the future.

In [ ]:
# Refit ARIMA model on all data and forecast future prices


# Refit selected ARIMA model on the full closing-price series
final_model = ARIMA(close, order=best_order_price)
final_fit = final_model.fit()

forecast_steps = 5

# Forecast 5 business/trading days ahead
future_forecast = np.asarray(final_fit.forecast(steps=forecast_steps))

# Create future business-day dates
future_dates = pd.bdate_range(
 start=dates.iloc[-1] + pd.Timedelta(days=1),
 periods=forecast_steps
)

future_forecast_table = pd.DataFrame({
 "Forecast date": future_dates,
 "Forecast closing price [USD]": future_forecast
})

display(future_forecast_table)

print("1-day-ahead forecast:")
print(f"${future_forecast[0]:.2f}")

print("\n5-day-ahead forecast:")
print(f"${future_forecast[-1]:.2f}")

In [ ]:
# Plot final future forecast

plt.figure()
plt.plot(dates, close, label="Historical platinum closing price")
plt.plot(future_dates, future_forecast, "o--", label="Future forecast")
plt.axvline(dates.iloc[-1], linestyle=":", label="End of available data")
plt.xlabel("Date")
plt.ylabel("Closing price [USD]")
plt.title("Final Platinum Closing-Price Forecast")
plt.legend()
plt.show()

# Zoomed view
plt.figure()
plt.plot(dates.iloc[-60:], close[-60:], label="Recent historical closing price")
plt.plot(future_dates, future_forecast, "o--", label="Future forecast")
plt.axvline(dates.iloc[-1], linestyle=":", label="End of available data")
plt.xlabel("Date")
plt.ylabel("Closing price [USD]")
plt.title("Zoomed Final Platinum Forecast")
plt.legend()
plt.show()

### Final Forecast Discussion

After the walk-forward validation, the selected ARIMA model was refitted using all available platinum closing-price data. This was done because the final forecast should use the maximum amount of historical information.

The final 1-day-ahead forecast was:

$$
\hat{x}_{t+1} = 1969.98 \text{ USD}
$$

The final 5-day-ahead forecast was:

$$
\hat{x}_{t+5} = 1958.66 \text{ USD}
$$

The forecast remains close to the most recent platinum closing price and shows a slight downward movement over the next few trading days. This is reasonable because the recent historical data is moving around the 1900 USD to 2000 USD range.

The final forecast does not show an unrealistic increase or decrease. This supports the use of the ARIMA model over the earlier polynomial extrapolation approach.

## 16. Actual Future Price Comparison

The final ARIMA forecast was compared with actual XPT/USD closing prices after the end of the dataset.

The dataset ends on 5 May 2026. Therefore, the 1-trading-day-ahead forecast corresponds to 6 May 2026, and the 5-trading-day-ahead forecast corresponds to 12 May 2026.

The forecast error is calculated as:

$$
e_{t+h} = x_{t+h} - \hat{x}_{t+h}
$$

The absolute error is:

$$
|e_{t+h}| = |x_{t+h} - \hat{x}_{t+h}|
$$

The percentage error is:

$$
\left|\frac{x_{t+h} - \hat{x}_{t+h}}{x_{t+h}}\right| \times 100
$$

In [ ]:
# Actual future forecast comparison


# Actual XPT/USD closing prices from Investing.com historical data
# Dates: May 06, May 07, May 08, May 11, May 12, 2026

actual_future_close = np.array([
 2064.33,
 2026.00,
 2058.86,
 2134.88,
 2130.30
])

future_accuracy = pd.DataFrame({
 "Date": future_dates,
 "Forecast close": future_forecast,
 "Actual close": actual_future_close
})

future_accuracy["Error"] = future_accuracy["Actual close"] - future_accuracy["Forecast close"]
future_accuracy["Absolute error"] = np.abs(future_accuracy["Error"])
future_accuracy["Percentage error"] = np.abs(
 future_accuracy["Error"] / future_accuracy["Actual close"]
) * 100

display(future_accuracy)

actual_future_mae = mean_absolute_error(actual_future_close, future_forecast)
actual_future_rmse = np.sqrt(mean_squared_error(actual_future_close, future_forecast))
actual_future_mape = np.mean(np.abs((actual_future_close - future_forecast) / actual_future_close)) * 100

print(f"Actual future MAE: ${actual_future_mae:.2f}")
print(f"Actual future RMSE: ${actual_future_rmse:.2f}")
print(f"Actual future MAPE: {actual_future_mape:.2f}%")

In [ ]:
# Plot actual future values against forecast values


plt.figure()
plt.plot(future_dates, future_forecast, "o--", label="Forecast closing price")
plt.plot(future_dates, actual_future_close, "x-", label="Actual closing price")
plt.xlabel("Date")
plt.ylabel("Closing price [USD]")
plt.title("Actual vs Forecasted Platinum Closing Price")
plt.legend()
plt.show()

### Actual Future Comparison Discussion

The final ARIMA forecast was compared with actual XPT/USD closing prices obtained after the end of the dataset.

The actual closing prices for the five forecasted trading days were:

$$
2064.33,\ 2026.00,\ 2058.86,\ 2134.88,\ 2130.30
$$

The forecast performance for these five future trading days was:

$$
MAE = 106.58 \text{ USD}
$$

$$
RMSE = 118.41 \text{ USD}
$$

$$
MAPE = 5.07\%
$$

The MAPE value means that the model's average forecast error over the five future trading days was approximately 5.07% of the actual platinum closing price.

The model predicted prices close to the 1950 USD to 2000 USD range, while the actual prices increased above 2100 USD by 11 May and 12 May. This caused the 5-day-ahead forecast error to be larger than the 1-day-ahead error.

Overall, the model gives a reasonable short-term statistical forecast, but the future comparison shows that platinum prices can still move away from the forecast due to market volatility and external factors.

## 17. Final Conclusion

The platinum closing-price series was analysed using stationarity testing, decomposition, train/test validation and ARIMA forecasting.

The original closing-price signal was found to be non-stationary, with an ADF p-value of 0.7426. Since this value is greater than 0.05, the null hypothesis was not rejected. After first differencing, the signal became stationary, with an ADF p-value of 0.0000. This justified the use of an ARIMA model with one order of differencing.

The time series was also decomposed into trend, seasonal and remainder components:

$$
x_t = T_t + S_t + R_t
$$

The decomposition showed that the platinum price contains a strong trend component, a smaller seasonal component and a volatile remainder component. The component-based approach was useful for analysing the structure of the signal. However, the component-based test forecast performed poorly because the polynomial trend extrapolated upward into the test period while the actual price decreased.

Therefore, an improved ARIMA model was fitted directly to the closing-price series. The selected model was:

$$
ARIMA(5,1,5)
$$

The model was validated using a walk-forward test forecast on unseen test data. The walk-forward validation gave the following performance:

$$
MAE = 66.61 \text{ USD}
$$

$$
RMSE = 98.37 \text{ USD}
$$

$$
MAPE = 3.23\%
$$

These results show that the ARIMA model followed the unseen test data reasonably well.

The final model was then refitted using all available data. The final forecasts were:

$$
\hat{x}_{t+1} = 1969.98 \text{ USD}
$$

and:

$$
\hat{x}_{t+5} = 1958.66 \text{ USD}
$$

The actual future closing prices were then used to evaluate the final forecast. The future comparison gave:

$$
MAE = 106.58 \text{ USD}
$$

$$
RMSE = 118.41 \text{ USD}
$$

$$
MAPE = 5.07\%
$$

The 5-day future MAPE of 5.07% shows that the model gave a reasonable short-term forecast, but it did not perfectly predict the upward movement in the actual future prices. This confirms that platinum prices are difficult to forecast accurately because they are affected by external factors such as supply and demand, mining activity, exchange rates, interest rates and market sentiment.

Based on this model alone, it would not be prudent to trade in platinum without further analysis. The model provides useful short-term statistical guidance, but a trading decision should also include updated market information, risk management and economic analysis.